# Programming Assignment-2
The goal of this assingment is to allow you to practice several the following things in Python:
1. Perfoming typical data processing (or preprocessing if you prefer). This includes all the typical data wraning such as creating news variables, combining several datasets and more 
2. Running explolatory data analysis including basic plotting of variables 
3. Perfoming basic inferential statisticals using statsmodels and scipy to run hypythesis testing and build simple statistial or econometric models.

## Datasets 
For this assignment, you will use the following datasets:
### Rwanda Health Indicators
The Excel file was generated by combining multiple CSV files, each containing data on different health indicators for Rwanda, So that each sheet in the file represent one such indicator. See below some of the input files which were used:
- `access-to-health-care_subnational_rwa`
- `child-mortality-rates_subnational_rwa`
- `dhs-mobile_subnational_rwa`

You can download the dataset from [here](https://docs.google.com/spreadsheets/d/1uvTQYS22VfXXo1Hwkm1frFx_bKkLQkcf/edit?usp=share_link&ouid=113302179168925233984&rtpof=true&sd=true).
### Nights lights Data
- Please download it [here](https://drive.google.com/file/d/1f_4fiqxIejly0YmC088s9bxOfrABv9Sz/view?usp=sharing) and check the documentation in the cells below. 

### Popupation Dataset
- Please download it [here](https://drive.google.com/file/d/1FWEFGdN-xDuFH1jmt0hr4F8Xc3Y5XzvB/view?usp=share_link) and check the documentation and metadata in the class notebooks.


## Submission Guidelines 
- Please guidelines and complete all steps in the [GitHub Workflow](https://dmatekenya.github.io/AIMS-DSCBI/course-requirements/github-workflow.html)
- Once you have completed your assignment, push chanegs to your repository.
- Send a link (copy from within GitHub) to your notebook to the tutors/teaching assistants


# Import Required Packages

In [55]:
from pathlib import Path
import pandas as pd
import numpy as np

# Setup Input Folders

As usual, it is good practice to set up input folders using the [`pathlib`](https://docs.python.org/3/library/pathlib.html) package. In this section, make sure to define the folders where your data is stored on your machine.

I find it helpful to set up the working directory and input data folders right at the start of the notebook. To keep things organized, I use the naming convention: `FILE_{NAME}` for files and `DIR_{NAME}` for folders. We use capital letters because these are global variables that will be referenced throughout the notebook.

We'll be using the [`pathlib`](https://docs.python.org/3/library/pathlib.html) library, which offers several advantages over traditional string-based path handling:

- **Cross-platform compatibility** - automatically handles path separators (`/` vs `\`) across different operating systems
- **Object-oriented approach** - paths are objects with useful methods rather than strings
- **Intuitive syntax** - use `/` operator to join paths naturally: `parent_dir / "subfolder" / "file.txt"`
- **Built-in path operations** - methods like `.exists()`, `.is_file()`, `.parent`, `.stem`, and `.suffix`
- **Safer path manipulation** - reduces errors from manual string concatenation and splitting

This is the recommended approach for managing file paths in modern Python development.


In [56]:
# Uncomment the following lines and add your code to define the directories and files
DIR_DATA = Path.cwd().parents[1].joinpath("data")
FILE_HEALTH_DATA = DIR_DATA/"RW-Health-Data.xlsx"
FILE_CELL_POP_DATA = DIR_DATA/"rwa-cell-pop.csv"
FILE_CELL_LIGHT_DATA = DIR_DATA/"cell-ntl-2015-2020-2024.csv"

# Part 1: Processing Excel Files
The primary goal is to preprocess an [Excel file](https://docs.google.com/spreadsheets/d/1uvTQYS22VfXXo1Hwkm1frFx_bKkLQkcf/edit?usp=share_link&ouid=113302179168925233984&rtpof=true&sd=true) with multiple sheets into a unified CSV dataset that consolidates multiple indicators. Having all indicators in a single file at the same analytical unit (national, subnational) is more efficient than managing separate files and enables easier cross-indicator analysis.

## Task 1: Generate National-Level Summaries

For each indicator, compute a single national-level value using appropriate aggregation functions such as **mean**, **sum** or **count**. For this one, all available indicators can be summarized at national level, so we will have a CSV file with one row and 

### Expected Output Structure
1. **DataFrame display** in Jupyter Notebook
2. **CSV file** with columns:
- `indicator_name`: Name of the indicator
- `aggregated_value`: Computed national value
- `indicator_year`: Survey year or something similar
- `survey_name`: Name of the survey where information is coming from
- `aggregation_method`: Statistical method used (optional)

## Task 2: Subnational-Level Indicator Dataset

Create a merged dataset for indicators with subnational data (ADM2/ADM3 levels), ensuring spatial alignment and consistent administrative boundaries.

### Expected Output Structure
   - `indicator_name`: Name of the indicator
   - `aggregated_value`: Computed national value
   - `indicator_year`: Survey year or something similar
   - `survey_name`: Name of the survey where information is coming from
   - `aggregation_method`: Statistical method used (optional)

This structure enables both single-indicator and multi-indicator analysis at the subnational level.

In [ ]:
sheet_names_health_df = pd.ExcelFile(FILE_HEALTH_DATA).sheet_names
excel_file = pd.ExcelFile(FILE_HEALTH_DATA)
combined_rwanda_health_indicators = pd.concat(
    [excel_file.parse(sheet).iloc[1:] for sheet in sheet_names_health_df],
    ignore_index=True
)

cell_population_data = pd.read_csv(FILE_CELL_POP_DATA)
cell_light_data = pd.read_csv(FILE_CELL_LIGHT_DATA)


In [58]:
combined_rwanda_health_grouped_indicators = (
    combined_rwanda_health_indicators
    .groupby(['Indicator', 'SurveyYear', 'SurveyType'])['Value']
    .agg('mean')
    .reset_index()
)
combined_rwanda_health_grouped_indicators["aggregation_method"] = "mean"
combined_rwanda_health_grouped_indicators = combined_rwanda_health_grouped_indicators.rename(
    columns={"Indicator": "indicator_name", 
             "SurveyYear": "indicator_year", 
             "SurveyType": "survey_name", 
             "Value": "aggregated_value"})
combined_rwanda_health_grouped_indicators["aggregated_value"] = pd.to_numeric(
    combined_rwanda_health_grouped_indicators["aggregated_value"], errors="coerce").round(2)


In [59]:
print(combined_rwanda_health_grouped_indicators)
combined_rwanda_health_grouped_indicators.to_csv(DIR_DATA / "combined_rwanda_health_grouped_indicators.csv", index=False)

                                         indicator_name  indicator_year  \
0     Accepting attitudes towards those living with ...            2005   
1     Accepting attitudes towards those living with ...            2010   
2     Accepting attitudes towards those living with ...            2015   
3     Accepting attitudes towards those living with ...            2005   
4     Accepting attitudes towards those living with ...            2010   
...                                                 ...             ...   
1450           Women with secondary or higher education            2010   
1451           Women with secondary or higher education            2013   
1452           Women with secondary or higher education            2015   
1453           Women with secondary or higher education            2017   
1454           Women with secondary or higher education            2019   

     survey_name  aggregated_value aggregation_method  
0            DHS             48.66         

In [60]:
pivot_df = combined_rwanda_health_grouped_indicators.pivot(
    index=["indicator_name", "survey_name", "aggregation_method"],  
    columns="indicator_year",                       
    values="aggregated_value"                    
).reset_index()

print(pivot_df)

indicator_year                                     indicator_name survey_name  \
0               Accepting attitudes towards those living with ...         DHS   
1               Accepting attitudes towards those living with ...         DHS   
2                              Age specific fertility rate: 15-19         DHS   
3                              Age specific fertility rate: 15-19         MIS   
4                              Age specific fertility rate: 20-24         DHS   
..                                                            ...         ...   
318                                         Women with any anemia         DHS   
319                                       Women with no education         DHS   
320                                       Women with no education         MIS   
321                      Women with secondary or higher education         DHS   
322                      Women with secondary or higher education         MIS   

indicator_year aggregation_

In [61]:
pivot_df.to_csv(DIR_DATA / "pivot_df.csv", index=False)

## Introduction to Nightlights Dataset

## What is Nightlight Data?

Nightlight data is satellite imagery capturing artificial light emissions from Earth's surface during nighttime. Satellites like VIIRS collect this data regularly, providing an **objective, real-time measure of human economic activity and development**.

### Raw Data: Radiance Measurements
The fundamental measurement in nightlight data is **radiance** - the amount of light energy detected by satellite sensors, measured in **nanowatts per square centimeter per steradian (nW/cm²/sr)**. Each pixel in satellite imagery contains a radiance value representing the light intensity from that specific location on Earth's surface.

### Annual Composite Generation
This dataset was created from **annual composite images** using VIIRS nightlight files for Rwanda. Annual composites are generated by:

- **Aggregating daily/monthly observations** throughout each year (2015, 2020, 2024)
- **Filtering out temporary light sources** (fires, lightning, aurora)
- **Removing cloud-affected observations** to ensure clear measurements
- **Averaging or taking median values** to create stable, representative annual measurements
- **Masking techniques** to exclude areas with unreliable data

The files used include both **average composites** (`average_masked`) and **median composites** (`median_masked`), with **cloud-free versions** (`vcmslcfg`) preferred over cloud-inclusive versions (`vcmcfg`) for more accurate measurements.

### Why Use Nightlight Data?

- **Consistent global coverage** - Available everywhere, regardless of local data quality
- **Real-time updates** - More current than traditional economic statistics
- **Objective measurement** - Not subject to reporting biases
- **High resolution** - Captures local development patterns
- **Proxy for development** - Light intensity correlates with economic activity, infrastructure, and quality of life

## Dataset Overview 

- **6,507 observations** across Rwanda's administrative cells
- **Three time periods**: 2015, 2020, 2024
- **Cell-level data** - Rwanda's smallest administrative units
- Allows temporal analysis of development trends

---

## Variable Definitions

### Administrative Identifiers
- **`cell_id`** - Unique identifier for linking with other datasets
- **`province_name`** - Province (5 total in Rwanda)
- **`district_name`** - District (30 total in Rwanda) 
- **`sector_name`** - Administrative level between district and cell
- **`cell_name`** - Specific cell name

### Core Nightlight Measurements

#### `total_nightlight`
- **Sum of all radiance values** within cell boundaries
- **Key indicator** of overall economic activity/development
- Higher values = more total development

#### `mean_nightlight` 
- **Average radiance** per pixel
- Indicates development intensity regardless of cell size
- Useful for comparing cells of different areas

#### `median_nightlight`
- **Middle radiance value** of all pixels (less sensitive to outliers)
- Better represents typical lighting in unevenly developed areas

#### `max_nightlight`
- **Highest radiance** within cell
- Indicates major infrastructure (hospitals, commercial centers)

#### `min_nightlight` & `std_nightlight`
- Minimum radiance and standard deviation
- High std = uneven development within cell

### Spatial Coverage Indicators

#### `pixel_count`
- **Total pixels** in cell (indicates geographic size)
- Used to normalize other measurements

#### `lit_pixel_count`
- **Number of pixels with detectable light** (radiance > 0)
- Shows spatial extent of development

#### `lit_pixel_percentage`
- **Percentage of cell area with lighting**
- Formula: `(lit_pixel_count ÷ pixel_count) × 100`
- **0% = completely dark, 100% = fully developed**

#### `year`
- Time period: 2015, 2020, or 2024

# Part-2: Demographic and Nightlights Data

## Part A: Varible Generation and Data Integration

### Population Dataset Variables (`rwa-cell-pop.csv`):
Create the following derived variables:
- **`dependency_ratio`** - `(children_under_five_2020 + elderly_60_plus_2020) / working_age_population * 100`
- **`people_per_building`** - `general_2020 / building_count`
- **`working_age_population`** - `general_2020 - children_under_five_2020 - elderly_60_plus_2020`
- **`infrastructure_index`** - Your own formula that incorporates `people_per_building` and other relevant variables to measure infrastructure adequacy. Document and justify your `infrastructure_index` methodology, explaining how `people_per_building` and other variables contribute to measuring infrastructure pressure.

### Nightlight Dataset Variables (`cell-ntl-2015-2020-2024.csv`):
Create the following temporal and development indicators:
- **`nightlight_change_2015_2024`** - Percentage change in total nightlight from 2015 to 2024
- **`mean_nightlight_change_2015_2024`** - Percentage change in mean nightlight from 2015 to 2024
- **`lit_pixel_percentage`** - Use existing or calculate: `(lit_pixel_count / pixel_count) * 100`

### Data Integration:
Merge the datasets using the appropriate column. 

## Part B: Exploratory Data Analysis

### Correlation Analysis:
1. **Correlation Heatmap**: Create a heatmap showing correlations between 10 key variables (mix of demographic, infrastructure, and nightlight variables). 
2. **Report the top 3 variable pairs** with the highest correlations and interpret their relationships.
3. **Identify unexpected correlations** and discuss potential explanations.

### Nightlight Trend Analysis:
1. **District Ranking**: Report the **top 5 districts** with the highest nightlight growth (2015-2024) and **bottom 5 districts** with the most decline or lowest growth.
2. **Lit Pixel Analysis**: Compare these districts using `lit_pixel_percentage` changes to understand whether growth represents intensification or spatial expansion.
3. **Create visualizations** showing nightlight trends for these extreme districts.

## Part C: Modeling

### Multivariate Linear Regression:
1. **Model Development**: Build a multivariate linear regression model predicting **population density** using both demographic and nightlight variables as predictors. Explore as many variables as possible at the beginning.
2. **Variable Selection**: Test different combinations of variables and report the **top 3 most predictive variables** of population density.
3. **Model Evaluation**: Report R-squared, coefficients, and statistical significance. Interpret what these results tell us about population-infrastructure relationships.



## Notes and Other Requirements
Please follow the genral guidelines below when preparing your analysis..

### Statistical Analysis:
- Properly handle missing data and outliers
- Use appropriate statistical tests and report p-values
- Calculate and interpret correlation coefficients
- Validate regression assumptions (normality, homoscedasticity)

### Data Management:
- Document all data cleaning and aggregation steps using markdown 
- Ensure consistent district naming across datasets

### Visualization Standards:
- Create clear, publication-quality heatmaps with appropriate color scales
- Design effective time series plots for nightlight trends
- Include proper axis labels, titles, and legends
- Use consistent formatting across all visualizations

### Reporting Requirements:
- Clearly state the top 3 most predictive variables with statistical justification
- Provide ranked lists for nightlight growth districts with supporting metrics
- Include model performance statistics and interpretation
- Document all methodological choices and assumptions

In [83]:
cell_population_data['working_age_population'] = (
    cell_population_data['general_2020'] - 
    (cell_population_data['children_under_five_2020'] + cell_population_data['elderly_60_plus_2020'])
)
cell_population_data['dependency_ratio'] = (
    (cell_population_data['children_under_five_2020'] + cell_population_data['elderly_60_plus_2020']) /
    (cell_population_data['working_age_population'])
) * 100

# cell_population_data['people_per_building'] = (cell_population_data['general_2020'] / cell_population_data['building_count'])


cell_population_data["people_per_building"] = np.where(
    cell_population_data["building_count"] > 0,
    cell_population_data["general_2020"] / cell_population_data["building_count"],
    cell_population_data["general_2020"]  
)


In [84]:
(cell_population_data["building_count"] == 0).sum()


1

In [89]:
def normalize(predictor_column, choice=True):
    normalized_values = (predictor_column - predictor_column.min()) / (predictor_column.max() - predictor_column.min())
    if choice==True:
        return normalized_values
    else:
        return 1 - normalized_values
    

cell_population_data["building_count_norm"] = normalize(cell_population_data["building_count"], choice=True)
cell_population_data["people_per_building_norm"] = normalize(cell_population_data["people_per_building"], choice=False)
cell_population_data["dependency_ratio_norm"] = normalize(cell_population_data["dependency_ratio"], choice=False)

cell_population_data["infrastructure_index"] = (
    0.4 * cell_population_data["building_count_norm"] +
    0.4 * cell_population_data["people_per_building_norm"] +
    0.2 * cell_population_data["dependency_ratio_norm"]
)*100

In [121]:

df_filtered = cell_light_data[cell_light_data['year'].isin([2015, 2024])]

df_pivot = df_filtered.pivot(
    index=["cell_id", "cell_name", "dist_name", "lit_pixel_percentage"], 
    columns="year",
    values=["total_nightlight", "mean_nightlight"]
).reset_index()


df_pivot.columns = [
    f"{col[0]}_{col[1]}" if col[1] != '' else col[0]
    for col in df_pivot.columns.to_flat_index()
]

df_pivot['nightlight_change_2015_2024'] = ( (df_pivot['total_nightlight_2024'] - df_pivot['total_nightlight_2015']) / df_pivot['total_nightlight_2015']) * 100
df_pivot['mean_nightlight_change_2015_2024'] = ( (df_pivot['mean_nightlight_2024'] - df_pivot['mean_nightlight_2015']) / df_pivot['mean_nightlight_2015']) * 100

df_pivot.head()

,cell_id,cell_name,dist_name,lit_pixel_percentage,total_nightlight_2015,total_nightlight_2024,mean_nightlight_2015,mean_nightlight_2024,nightlight_change_2015_2024,mean_nightlight_change_2015_2024
0,RWA.1.1.1.1_1,Bungwe,Burera,37.142857,1.900350,11.608531,0.027148,0.165836,510.862720,510.862733
1,RWA.1.1.1.2_1,Bushenya,Burera,42.857143,1.536613,10.771848,0.024391,0.170982,601.012355,601.012330
2,RWA.1.1.1.3_1,Mudugari,Burera,43.333333,1.445172,10.102386,0.024086,0.168373,599.043610,599.043614
3,RWA.1.1.1.4_1,Tumba,Burera,43.434343,2.817203,17.876629,0.028457,0.180572,534.552323,534.552310
4,RWA.1.1.10.1_1,Bugamba,Burera,46.000000,3.146631,20.117586,0.031466,0.201176,539.337363,539.337358


In [124]:
pop_light_merged = pd.merge(cell_population_data, cell_light_data, on='cell_id', how='left')
pop_light_merged

,cell_id,province_name_x,district_name_x,sector_name_x,cell_name_x,elderly_60_plus_2020,general_2020,children_under_five_2020,youth_15_24_2020,men_2020,...,total_nightlight,mean_nightlight,median_nightlight,max_nightlight,min_nightlight,std_nightlight,pixel_count,lit_pixel_count,year,lit_pixel_percentage
0,RWA.1.1.1.1_1,Amajyaruguru,Burera,Bungwe,Bungwe,241.693282,3855.623385,495.422606,758.093936,1850.711053,...,6.994746,0.099925,0.0,0.300291,0.0,0.130675,70,26,2020,37.142857
1,RWA.1.1.1.1_1,Amajyaruguru,Burera,Bungwe,Bungwe,241.693282,3855.623385,495.422606,758.093936,1850.711053,...,11.608531,0.165836,0.0,0.586523,0.0,0.218413,70,26,2024,37.142857
2,RWA.1.1.1.1_1,Amajyaruguru,Burera,Bungwe,Bungwe,241.693282,3855.623385,495.422606,758.093936,1850.711053,...,1.900350,0.027148,0.0,0.142328,0.0,0.038462,70,26,2015,37.142857
3,RWA.1.1.1.2_1,Amajyaruguru,Burera,Bungwe,Bushenya,229.611624,3669.128833,470.655098,720.896415,1761.457437,...,7.167736,0.113774,0.0,0.307715,0.0,0.131960,63,27,2020,42.857143
4,RWA.1.1.1.2_1,Amajyaruguru,Burera,Bungwe,Bushenya,229.611624,3669.128833,470.655098,720.896415,1761.457437,...,10.771848,0.170982,0.0,0.463222,0.0,0.198171,63,27,2024,42.857143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6502,RWA.5.3.10.3_1,UmujyiwaKigali,Nyarugenge,Rwezamenyo,RwezamenyoI,81.618488,2913.380392,298.982254,801.811389,1507.543042,...,42.085358,4.676151,0.0,21.477812,0.0,8.750682,9,2,2024,22.222222
6503,RWA.5.3.10.3_1,UmujyiwaKigali,Nyarugenge,Rwezamenyo,RwezamenyoI,81.618488,2913.380392,298.982254,801.811389,1507.543042,...,22.150925,2.461214,0.0,11.550793,0.0,4.609958,9,2,2015,22.222222
6504,RWA.5.3.10.4_1,UmujyiwaKigali,Nyarugenge,Rwezamenyo,RwezamenyoIi,56.925037,2285.696907,279.887387,583.358607,1155.323286,...,31.468475,5.244746,0.0,20.264875,0.0,7.864919,6,2,2020,33.333333
6505,RWA.5.3.10.4_1,UmujyiwaKigali,Nyarugenge,Rwezamenyo,RwezamenyoIi,56.925037,2285.696907,279.887387,583.358607,1155.323286,...,42.503082,7.083847,0.0,23.240290,0.0,10.083657,6,2,2024,33.333333


In [123]:
import pandas as pd

# df1
df1 = pd.DataFrame({
    'cell_id': [1, 2, 3],
    'total_nightlight_2015': [100, 50, 70]
})

# df2
df2 = pd.DataFrame({
    'cell_id': [1, 2, 3],
    'total_nightlight_2024': [200, 70, 90]
})
df_merged = pd.merge(df1, df2, on='cell_id', how='left')
df_merged

,cell_id,total_nightlight_2015,total_nightlight_2024
0,1,100,200
1,2,50,70
2,3,70,90
